# /classify — Comprehensive Evaluation

## How /classify works

```
summary (text)
    │
    ├─ Relevance gate  (XNLI single hypothesis)
    │   └─ Score < threshold → out_of_scope=True, stop
    │
    ├─ Topic NLI  (XNLI multi-label against taxonomy)
    │   └─ Blacklist checked; top score > threshold → out_of_scope=True, stop
    │
    ├─ Topic filtering  →  labels above score_threshold, top-k kept
    │
    └─ Scope
        ├─ geo_scope provided (from /geotag) → use directly, skip NLI
        └─ geo_scope absent → own 3-way NLI (city / regional / national)
```

**Taxonomy:** infraestructura ciclista, carril bici, accidente de tráfico, presupuesto municipal, movilidad sostenible, regulación de movilidad, evento de movilidad, contaminación urbana, peatonalización, obras de movilidad, bicicleta eléctrica, aparcamiento de bicicletas, tráfico rodado, seguridad vial, transporte activo, patinete eléctrico, zonas de bajas emisiones, política de movilidad, urbanismo táctico, opinión y debate

**Sections:**
1. Out-of-scope rejection — blacklisted article types must be caught
2. In-scope topic assignment — accepted articles must get correct topic labels
3. Edge cases — borderline articles that test the boundary
4. Scope classification — with and without geo_scope passthrough

**Prerequisite:** NLP service running. `/readyz` → 200.

In [ ]:
import sys, time, requests
sys.path.insert(0, '.')
from _scorecard import load_fixture, print_scorecard, keyword_hit_rate, NLP_BASE_URL, HEADERS

cases = load_fixture('classify_cases.json')
print(f'Loaded {len(cases)} cases')

def call_classify(text, geo_scope=None, geo_cities=None):
    t0 = time.monotonic()
    resp = requests.post(
        f'{NLP_BASE_URL}/classify',
        json={
            'article_id':   'eval',
            'summary':      text,
            'geo_cities':   geo_cities or [],
            'search_tags':  [],
            'source_profile': None,
            'geo_scope':    geo_scope,
        },
        headers=HEADERS,
    )
    latency = time.monotonic() - t0
    assert resp.status_code == 200, f'HTTP {resp.status_code}: {resp.text}'
    return resp.json(), latency

In [ ]:
r = requests.get(f'{NLP_BASE_URL}/readyz', headers=HEADERS)
assert r.status_code == 200, f'Service not ready: {r.status_code} {r.text}'
print('Service ready')

## Section 1 — Out-of-Scope Rejection

Articles that should be flagged .
Two rejection paths: **relevance gate** (score < 0.4) or **blacklist** (score > 0.7).
A ✗ here means a false negative — the classifier accepted an article it should have rejected.

In [ ]:
oos_cases = [c for c in cases if c.get('expected_out_of_scope') is True]
oos_results = []

print(f'Out-of-scope cases: {len(oos_cases)}')
print()

for case in oos_cases:
    data, latency = call_classify(case['text'])
    is_oos = data['out_of_scope']
    ok     = is_oos == True
    icon   = '✓' if ok else '✗'

    print(f'  {icon} [{case["article_id"]}]  {case["description"]}')
    print(f'    text   : {case["text"][:120]}')
    print(f'    result : out_of_scope={is_oos}  topics={data["topics"]}  scope={data["geo_scope"]!r}')
    if not ok:
        print(f'    ✗ FALSE NEGATIVE — accepted but should be rejected')
        if case.get('notes'):
            print(f'    note: {case["notes"]}')
    print(f'    latency: {latency:.2f}s')
    print()
    oos_results.append(ok)

print(f'Out-of-scope rejection: {sum(oos_results)}/{len(oos_results)} correctly rejected')

## Section 2 — In-Scope Topic Assignment

Articles that should be accepted and assigned the correct topic labels.
A ✗ here means a false positive (rejected when it should be accepted) or wrong topics.

In [ ]:
in_scope_cases = [c for c in cases if c.get('expected_out_of_scope') is False
                  and c.get('expected_topics_include')]
topic_results = []

print(f'In-scope cases: {len(in_scope_cases)}')
print()

for case in in_scope_cases:
    data, latency = call_classify(case['text'])
    is_oos = data['out_of_scope']

    if is_oos:
        print(f'  ✗ [{case["article_id"]}]  {case["description"]}')
        print(f'    FALSE POSITIVE — rejected (out_of_scope=True)')
        print(f'    text: {case["text"][:120]}')
        print(f'    latency: {latency:.2f}s')
        print()
        topic_results.append({'id': case['article_id'], 'accepted': False, 'topic_ok': False})
        continue

    expected_kws = case.get('expected_topics_include', [])
    topics_text  = ' '.join(data['topics'])
    hit_rate     = keyword_hit_rate(topics_text, expected_kws)
    topic_ok     = hit_rate >= 0.5 or not expected_kws
    icon         = '✓' if topic_ok else '✗'

    print(f'  {icon} [{case["article_id"]}]  {case["description"]}')
    print(f'    expected topics : {expected_kws}')
    print(f'    got topics      : {data["topics"]}')
    print(f'    keyword hit     : {hit_rate:.2f}')
    if not topic_ok and case.get('notes'):
        print(f'    note: {case["notes"]}')
    print(f'    latency: {latency:.2f}s')
    print()
    topic_results.append({'id': case['article_id'], 'accepted': True, 'topic_ok': topic_ok})

accepted = sum(1 for r in topic_results if r['accepted'])
topic_ok  = sum(1 for r in topic_results if r['topic_ok'])
print(f'In-scope: {accepted}/{len(topic_results)} accepted  |  {topic_ok}/{len(topic_results)} correct topics')

## Section 3 — Edge Cases

Borderline articles probing the in-scope/out-of-scope boundary.
Each has an explicit expected result and why it matters.

In [ ]:
edge_cases = [
    {
        'id': 'edge-01', 'expected_oos': False,
        'label': 'Park WITH cycling — should be in-scope',
        'text': 'El Parque de la Ciutadella abre un nuevo carril bici de 2 km que conecta con el carril de la Av. Marquès de l'Argentera. La mejora forma parte del plan de movilidad sostenible del distrito.',
        'expected_topics': ['carril bici'],
    },
    {
        'id': 'edge-02', 'expected_oos': False,
        'label': 'Metro mentioned secondary to cycling — should be in-scope',
        'text': 'El nuevo aparcamiento de bicicletas de la estación de metro de Nuevos Ministerios abrirá en septiembre con 300 plazas. La iniciativa busca fomentar la intermodalidad bici+metro en la ciudad.',
        'expected_topics': ['aparcamiento de bicicletas'],
    },
    {
        'id': 'edge-03', 'expected_oos': False,
        'label': 'Opinion/debate about urban mobility — should be in-scope',
        'text': 'La peatonalización del centro histórico divide a comerciantes y ciclistas. Mientras unos celebran la recuperación del espacio público, otros temen que la medida no vaya acompañada de suficiente infraestructura ciclista.',
        'expected_topics': ['peatonalización'],
    },
    {
        'id': 'edge-04', 'expected_oos': False,
        'label': 'Air quality + ZBE policy — should be in-scope',
        'text': 'Los niveles de NO2 en el centro de Madrid superaron el límite legal durante 30 días en 2025. Expertos piden ampliar las zonas de bajas emisiones y restringir el tráfico en horas punta.',
        'expected_topics': ['zonas de bajas emisiones', 'contaminación urbana'],
    },
    {
        'id': 'edge-05', 'expected_oos': True,
        'label': 'Highway accident, no cycling — should be out-of-scope',
        'text': 'Un accidente en la A-6 implicó a tres camiones y provocó retenciones de 15 km durante la tarde del viernes. La DGT recomienda evitar esa vía durante las próximas horas.',
        'expected_topics': [],
    },
    {
        'id': 'edge-06', 'expected_oos': True,
        'label': 'Metro expansion as primary subject — should be out-of-scope',
        'text': 'Metro de Madrid ampliará la línea 11 con tres nuevas estaciones en el sur de la capital. Las obras, presupuestadas en 450 millones de euros, comenzarán en 2027 y estarán terminadas en 2031.',
        'expected_topics': [],
    },
    {
        'id': 'edge-07', 'expected_oos': True,
        'label': 'Urban park festival, no mobility — should be out-of-scope',
        'text': 'El Parque del Retiro celebra este fin de semana su festival anual de jardines, con exhibiciones de flores y plantas de más de 40 países. La entrada es gratuita para todos los visitantes.',
        'expected_topics': [],
    },
    {
        'id': 'edge-08', 'expected_oos': False,
        'label': 'Cyclist accident on bike lane — should be in-scope (safety)',
        'text': 'Un ciclista de 34 años resultó herido leve ayer por la tarde tras ser golpeado por un vehículo en el carril bici del Paseo del Prado. El conductor fue detenido por la Policía Municipal.',
        'expected_topics': ['accidente de tráfico', 'seguridad vial'],
    },
]

edge_results = []
print(f'Edge cases: {len(edge_cases)}')
print()

for case in edge_cases:
    data, latency = call_classify(case['text'])
    is_oos     = data['out_of_scope']
    exp_oos    = case['expected_oos']
    oos_ok     = is_oos == exp_oos

    topics_text = ' '.join(data['topics'])
    hit_rate    = keyword_hit_rate(topics_text, case.get('expected_topics', []))
    topic_ok    = hit_rate >= 0.5 or not case.get('expected_topics')

    passed = oos_ok and topic_ok
    icon   = '✓' if passed else '✗'

    print(f'  {icon} [{case["id"]}]  {case["label"]}')
    print(f'    expected oos={exp_oos}  got oos={is_oos}')
    if not exp_oos:
        print(f'    expected topics : {case.get("expected_topics", [])}')
        print(f'    got topics      : {data["topics"]}  (hit={hit_rate:.2f})')
    print(f'    latency: {latency:.2f}s')
    print()
    edge_results.append(passed)

print(f'Edge cases: {sum(edge_results)}/{len(edge_results)} correct')

## Section 4 — Scope Classification

**4a — With geo_scope passthrough:**  receives geo_scope from a fixture value.
The classifier must return exactly that scope (no NLI re-computation).

**4b — Without geo_scope:**  runs its own 3-way NLI.
Checks the classifier's independent scope against expected.

In [ ]:
# 4a: geo_scope passthrough
print('=== 4a: Scope passthrough (geo_scope injected) ===')
print()

scope_fixture = [
    {'id': 'scope-city',     'text': 'El Ayuntamiento de Sevilla ha inaugurado 15 kilómetros de nueva red ciclista en los barrios del norte de la ciudad.', 'geo_scope': 'city',     'expected_topics': ['infraestructura ciclista']},
    {'id': 'scope-regional', 'text': 'La Generalitat de Catalunya ha anunciado un plan de 80 millones de euros para crear una red ciclista entre las capitales de comarca de la provincia de Girona.', 'geo_scope': 'regional', 'expected_topics': ['infraestructura ciclista']},
    {'id': 'scope-national', 'text': 'El Gobierno de España ha aprobado el Plan Estatal de Movilidad Sostenible 2026-2035, que dotará con 2.000 millones de euros a municipios de todo el país para mejorar la infraestructura ciclista y peatonal.', 'geo_scope': 'national', 'expected_topics': ['política de movilidad']},
]

passthrough_results = []
for case in scope_fixture:
    data, latency = call_classify(case['text'], geo_scope=case['geo_scope'])
    cls_scope   = data['geo_scope']
    scope_match = cls_scope == case['geo_scope']
    icon        = '✓' if scope_match else '✗'

    print(f'  {icon} [{case["id"]}]  injected={case["geo_scope"]!r}  →  classify returned={cls_scope!r}')
    print(f'    topics={data["topics"]}  oos={data["out_of_scope"]}  latency={latency:.2f}s')
    print()
    passthrough_results.append(scope_match)

print(f'Passthrough: {sum(passthrough_results)}/{len(passthrough_results)} match')

In [ ]:
# 4b: own NLI scope (no passthrough)
print()
print('=== 4b: Own NLI scope (no geo_scope injected) ===')
print()

scope_cases_fixture = [c for c in cases if c.get('expected_scope_signal')]
scope_results_b = []

for case in scope_cases_fixture:
    data, latency = call_classify(case['text'], geo_scope=None)
    cls_scope = data['geo_scope']
    exp_scope = case['expected_scope_signal']
    scope_ok  = cls_scope == exp_scope
    icon      = '✓' if scope_ok else '✗'

    print(f'  {icon} [{case["article_id"]}]  {case["description"]}')
    print(f'    expected={exp_scope!r}  got={cls_scope!r}  oos={data["out_of_scope"]}  latency={latency:.2f}s')
    print()
    scope_results_b.append(scope_ok)

print(f'Own NLI scope: {sum(scope_results_b)}/{len(scope_results_b)} correct')

## Overall Scorecard

In [ ]:
print_scorecard('/classify comprehensive', {
    'S1 OOS rejection':          f'{sum(oos_results)}/{len(oos_results)}',
    'S2 In-scope accepted':      f'{accepted}/{len(topic_results)}',
    'S2 Correct topics':         f'{topic_ok}/{len(topic_results)}',
    'S3 Edge cases':             f'{sum(edge_results)}/{len(edge_results)}',
    'S4a Scope passthrough':     f'{sum(passthrough_results)}/{len(passthrough_results)}',
    'S4b Own NLI scope':         f'{sum(scope_results_b)}/{len(scope_results_b)}',
})